# Challenge 5 — Deploy an Agent to Agent Platform

This Gemini-only notebook creates a Google Search agent, tests it locally, deploys it to Google Agent Platform, and tests the deployed agent. It does not store API keys.

In [ ]:
%pip install -q --upgrade google-adk google-cloud-aiplatform[agent_engines,adk]

import os
import vertexai
from google.cloud import storage
from google.adk.agents import LlmAgent
from google.adk.tools import google_search
from vertexai.preview import reasoning_engines
from vertexai import agent_engines

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "qwiklabs-gcp-02-9e12deb8c42f")
LOCATION = "us-central1"
MODEL_GEMINI = "gemini-2.5-flash"
STAGING_BUCKET_NAME = f"{PROJECT_ID}-challenge5-staging"
STAGING_BUCKET = f"gs://{STAGING_BUCKET_NAME}"

storage_client = storage.Client(project=PROJECT_ID)
if storage_client.lookup_bucket(STAGING_BUCKET_NAME) is None:
    bucket = storage_client.bucket(STAGING_BUCKET_NAME)
    bucket.location = LOCATION
    storage_client.create_bucket(bucket, location=LOCATION)
    print(f"Created temporary staging bucket: {STAGING_BUCKET}")
else:
    print(f"Using existing staging bucket: {STAGING_BUCKET}")

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)
print(f"Initialized Vertex AI for {PROJECT_ID} in {LOCATION}.")

In [ ]:
deployment_agent = LlmAgent(
    name="preparedness_search_agent",
    model=MODEL_GEMINI,
    description="Finds current, reliable emergency-preparedness information.",
    instruction=(
        "Use Google Search for current information when it helps. Give a concise, "
        "well-structured answer, distinguish urgent safety advice from general preparation, "
        "and do not invent sources or emergency instructions."
    ),
    tools=[google_search],
)

print("Created the deployment agent.")

In [ ]:
# Local test before deployment.
local_app = reasoning_engines.AdkApp(agent=deployment_agent)
LOCAL_USER_ID = "challenge-five-local-tester"
LOCAL_QUESTION = "What are three reliable steps to prepare for a hurricane?"

local_session = local_app.create_session(user_id=LOCAL_USER_ID)
local_session_id = local_session["id"] if isinstance(local_session, dict) else local_session.id
print(f"=== Local test: {LOCAL_QUESTION} ===")

for event in local_app.stream_query(
    user_id=LOCAL_USER_ID,
    session_id=local_session_id,
    message=LOCAL_QUESTION,
):
    author = event.get("author", "unknown") if isinstance(event, dict) else getattr(event, "author", "unknown")
    content = event.get("content") if isinstance(event, dict) else getattr(event, "content", None)
    parts = content.get("parts", []) if isinstance(content, dict) else getattr(content, "parts", [])
    text = " ".join(part.get("text", "") if isinstance(part, dict) else getattr(part, "text", "") for part in parts).strip()
    if text:
        print(f"[{author}] {text}")

In [ ]:
# Deploy once. Save the resource name printed below so the remote test can be rerun without redeploying.
remote_agent = agent_engines.create(
    local_app,
    requirements=["google-cloud-aiplatform[agent_engines,adk]", "google-adk"],
)

REMOTE_AGENT_NAME = getattr(remote_agent, "resource_name", None) or getattr(remote_agent, "name", None)
print(f"Deployment complete: {REMOTE_AGENT_NAME}")

In [ ]:
# Test the deployed Agent Platform agent.
REMOTE_USER_ID = "challenge-five-remote-tester"
REMOTE_QUESTION = "Give a short hurricane-preparedness checklist for a family."

print(f"=== Deployed-agent test: {REMOTE_QUESTION} ===")
for event in remote_agent.stream_query(
    user_id=REMOTE_USER_ID,
    message=REMOTE_QUESTION,
):
    content = event.get("content") if isinstance(event, dict) else getattr(event, "content", None)
    parts = content.get("parts", []) if isinstance(content, dict) else getattr(content, "parts", [])
    text = " ".join(part.get("text", "") if isinstance(part, dict) else getattr(part, "text", "") for part in parts).strip()
    if text:
        print(text)

## Requirement checklist

- An ADK agent is created with Google Search.
- The agent is tested locally before deployment.
- `agent_engines.create()` deploys it to Agent Platform.
- The final cell tests the deployed agent and prints its response.